# 00 - Arranque

Ejecutar las celdas en orden en un servidor de Colab (CPU, RAM alta). Ver `README.md`.

In [1]:
import os
import sys

REPO = "Adriel23456/Applied-AI-Agent-Project"
BRANCH = "03DataCollectionAndEnvironmentSetup"

if not os.path.exists("/content/repo"):
    !git clone -b {BRANCH} https://github.com/{REPO}.git /content/repo
else:
    !git -C /content/repo pull

%cd /content/repo
!pip install -q -r requirements.txt
sys.path.insert(0, "/content/repo/src")

Already up to date.
/content/repo


In [3]:
from ptcg_wp.colab import setup_kaggle

setup_kaggle()
!kaggle datasets list -s "pokemon tcg ai battle" | head -5

ref                                               title                                                                                size  lastUpdated                 downloadCount  voteCount  usabilityRating  
------------------------------------------------  -----------------------------------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
busyaprime/pokemon-tcg-ai-battle-live-meta        Pokemon TCG AI Battle: live ladder meta                                              2614  2026-08-01 16:00:59.037000            858         13  0.64705884       
kaggle/pokemon-tcg-ai-battle-episodes-index       The Pokémon Company - PTCG AI Battle Challenge Simulation Episodes Index             2606  2026-08-31 00:13:57.020000         104375         36  0.4117647        
citerne/pokemon-tcg-agent-showcase-media          Pokemon TCG Agent Showcase Media                                                 29061831  2026-08

## Ingesta

Descarga cada día de `configs/dias.yaml`, escribe un Parquet por día y borra el crudo.

In [ ]:
!PYTHONPATH=src python -m ptcg_wp.ingest --config configs/dias.yaml --raw /content/raw --out /content/datos

In [ ]:
import pandas as pd
from pathlib import Path

d = Path("/content/datos")
games = pd.concat(pd.read_parquet(p) for p in sorted(d.glob("partidas_*.parquet")))
states = pd.concat(pd.read_parquet(p) for p in sorted(d.glob("estados_*.parquet")))

print("partidas:", games.shape)
print(games["winner"].value_counts())
print("estados:", states.shape, "| partidas con estados:", states["game_id"].nunique())
print("prevalencia y:", states["y"].mean().round(3))
print("filas setup:", states["setup"].sum(), "| anómalas:", states["anomalous"].sum())
print("MB en RAM:", round(states.memory_usage(deep=True).sum() / 1e6))